In [0]:
%run ./00_imports

In [0]:
print(f"Great Expectations version : {gx.__version__}")
print(f"Table cible : {TABLES['silver']}")

In [0]:
# ==============================================================================
# On travaille sur un échantillon représentatif avant de valider tout le dataset
# ==============================================================================

df_silver = spark.table(TABLES['silver'])
df_sample = df_silver.limit(10000)

print(f"Schéma Silver :")
df_silver.printSchema()
print(f"Lignes échantillon : {df_sample.count()}")

In [0]:

# ======================================================================
# EphemeralDataContext = contexte en mémoire, pas de fichiers sur disque
# ======================================================================

context = gx.get_context(mode="ephemeral")

# Conversion en pandas pour GX (GX 0.18 travaille nativement avec pandas)
df_pandas = df_sample.toPandas()

print(f"Contexte GX créé : {type(context)}")
print(f"Échantillon converti : {len(df_pandas)} lignes, {len(df_pandas.columns)} colonnes")
print(f"Colonnes : {list(df_pandas.columns)}")

In [0]:

# ==============================================================
# Création de la source de données et de la suite d'expectations
# Une suite = un ensemble de règles de validation groupées
# ==============================================================

data_source = context.sources.add_pandas("silver_source")
data_asset = data_source.add_dataframe_asset(name="silver_sample")
batch_request = data_asset.build_batch_request(dataframe=df_pandas)

suite = context.add_expectation_suite(expectation_suite_name="qualite_eau_suite")

print(f"Source créée : silver_source")
print(f"Asset créé : silver_sample")
print(f"Suite créée : qualite_eau_suite")

In [0]:

# ================================================================
# Chaque expectation = une règle métier vérifiable automatiquement
# ================================================================

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="qualite_eau_suite"
)

# --- Colonnes obligatoires non nulles ---
validator.expect_column_values_to_not_be_null("reference_prelevement")
validator.expect_column_values_to_not_be_null("code_parametre", mostly=0.995) # Autorise 0.5% de nulls (données source imparfaites)
validator.expect_column_values_to_not_be_null("conforme")
validator.expect_column_values_to_not_be_null("_annee")

# --- Types attendus ---
validator.expect_column_values_to_be_of_type("conforme", "bool")
validator.expect_column_values_to_be_of_type("valeur_numerique", "float")

# --- Valeurs attendues ---
# Années : plage ouverte >= 2023 pour évolutivité continue du dataset
validator.expect_column_values_to_be_between(
    "_annee", min_value=2023
)
validator.expect_column_values_to_be_between(
    "valeur_numerique", min_value=0, mostly=0.95
)

# --- Volumétrie ouverte pour s'adapter à tout échantillon ---
validator.expect_table_row_count_to_be_between(min_value=1000, max_value=200000)

validator.save_expectation_suite(discard_failed_expectations=False)
print("Expectations définies et sauvegardées")

In [0]:

# =========================================================================
# Le checkpoint lance toutes les expectations et produit un résultat global
# =========================================================================

checkpoint = context.add_or_update_checkpoint(
    name="checkpoint_silver",
    validator=validator
)

results = checkpoint.run()

# Résultat global
validation_result = list(results.run_results.values())[0]
stats = validation_result["validation_result"]["statistics"]

print(f"Résultat global : {'SUCCÈS' if results.success else 'ÉCHEC'}")
print(f"Expectations évaluées : {stats['evaluated_expectations']}")
print(f"Réussies : {stats['successful_expectations']}")
print(f"Échouées : {stats['unsuccessful_expectations']}")
print(f"Taux de succès : {stats['success_percent']:.1f}%")

In [0]:

# ====================================================
# Affiche le résultat de chaque règle individuellement
# ====================================================

resultats_detail = validation_result["validation_result"]["results"]

print(f"{'Expectation':<55} {'Résultat':<10} {'Détail'}")
print("-" * 100)

for r in resultats_detail:
    expectation_type = r["expectation_config"]["expectation_type"]
    colonne = r["expectation_config"]["kwargs"].get("column", "table")
    succes = "✅ OK" if r["success"] else "❌ ÉCHEC"
    detail = ""
    if "unexpected_percent" in r["result"]:
        detail = f"{r['result']['unexpected_percent']:.2f}% valeurs inattendues"
    print(f"{expectation_type} [{colonne}]  {succes}  {detail}")

#### ⚠️ Problème de validation sur le dataset Silver complet

La tentative de validation sur le dataset **Silver complet** échoue avec une erreur de mémoire JVM (**Out of Memory**).

---

###### 🛑 Cause du problème

Le code tente de charger l'intégralité du dataset Silver (**38M lignes**) en mémoire avec :

python
###### ❌ À éviter : chargement complet en mémoire (risque Out of Memory)
df_full_pandas = spark.table(TABLES['silver']).toPandas()


---
###### ❓ Pourquoi un échantillonnage stratifié plutôt que le dataset complet ?

Great Expectations 0.18 repose sur un backend **Pandas** pour exécuter les validations.
La conversion `toPandas()` charge l'intégralité du dataset en mémoire du driver Spark.

Sur le serverless Databricks Free Edition, cette mémoire est limitée :
- **38M lignes × 23 colonnes** → dépasse la capacité disponible → **Out of Memory JVM**

---
###### ✅ Stratégie retenue : échantillonnage stratifié par année

- **100 000 lignes** réparties proportionnellement sur toutes les années disponibles
- Le paramètre `seed=42` garantit la reproductibilité de l'échantillon
- Les résultats sur 10K lignes (cellule 3) étant déjà probants à 100%, 100K lignes offrent une confirmation statistiquement solide sans risque mémoire

---

> Cette contrainte est documentée comme limitation connue de l'environnement et sera mentionnée dans le rapport final.

---

In [0]:

# =======================================================================
# toPandas() sur 38M lignes dépasse la mémoire driver du serverless
# Stratégie : échantillon représentatif de 100K lignes, réparti par année
# =======================================================================

df_silver_full = spark.table(TABLES['silver'])

# Échantillonnage stratifié : fraction uniforme par année
fractions = {annee: 100000 / df_silver_full.count()
             for annee in [2023, 2024, 2025, 2026]}
df_stratifie = df_silver_full.sampleBy("_annee", fractions=fractions, seed=42)

df_full_pandas = df_stratifie.toPandas()
print(f"Échantillon stratifié : {len(df_full_pandas)} lignes")
print(f"Répartition par année :")
print(df_stratifie.groupBy("_annee").count().orderBy("_annee").display())

In [0]:
# =====================================================================
# Validation de l'échantillon stratifié (100K lignes réparties par année)
# =====================================================================

# Réutilisation de la source existante avec un nouvel asset
data_asset_full = data_source.add_dataframe_asset(name="silver_stratified")
batch_request_full = data_asset_full.build_batch_request(dataframe=df_full_pandas)

# Utilisation de la même suite d'expectations
validator_full = context.get_validator(
    batch_request=batch_request_full,
    expectation_suite_name="qualite_eau_suite"
)

# Exécution du checkpoint
checkpoint_full = context.add_or_update_checkpoint(
    name="checkpoint_silver_stratified",
    validator=validator_full
)

results_full = checkpoint_full.run()

# Extraction des statistiques
validation_result_full = list(results_full.run_results.values())[0]
stats_full = validation_result_full["validation_result"]["statistics"]

print(f"Résultat global (échantillon stratifié) : {'SUCCÈS' if results_full.success else 'ÉCHEC'}")
print(f"Expectations évaluées : {stats_full['evaluated_expectations']}")
print(f"Réussies : {stats_full['successful_expectations']}")
print(f"Échouées : {stats_full['unsuccessful_expectations']}")
print(f"Taux de succès : {stats_full['success_percent']:.1f}%")

In [0]:

# ================================================================
# Synthèse de la validation qualité
# Rapport lisible pour le rapport final et la fiche méthodologique
# ================================================================


print("=" * 60)
print("RAPPORT DE VALIDATION — GREAT EXPECTATIONS")
print("=" * 60)
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Table validée : {TABLES['silver']}")
print(f"Version GX : {gx.__version__}")
print()
print("--- ÉCHANTILLON (10 000 lignes) ---")
print(f"Expectations : {stats['evaluated_expectations']}")
print(f"Réussies     : {stats['successful_expectations']}")
print(f"Taux         : {stats['success_percent']:.1f}%")
print()
print("--- ÉCHANTILLON STRATIFIÉ (100K lignes, réparti par année) ---")
print(f"Expectations : {stats_full['evaluated_expectations']}")
print(f"Réussies     : {stats_full['successful_expectations']}")
print(f"Taux         : {stats_full['success_percent']:.1f}%")
print()
print("--- RÈGLES VALIDÉES ---")
print("  - Colonnes non nulles : reference_prelevement, code_parametre, conforme, _annee")
print("  - Types corrects : conforme (bool), valeur_numerique (float)")
print("  - Années >= 2023 (dataset évolutif)")
print("  - valeur_numerique >= 0 sur 95% des lignes")
print("  - Volumétrie cohérente")
print("=" * 60)

In [0]:
# ================================================================
# Même affichage que cellule 8, appliqué à l'échantillon stratifié
# ================================================================

resultats_detail_full = list(results_full.run_results.values())[0]["validation_result"]["results"]

print(f"{'Expectation':<55} {'Résultat':<10} {'Détail'}")
print("-" * 100)

for r in resultats_detail_full:
    expectation_type = r["expectation_config"]["expectation_type"]
    colonne = r["expectation_config"]["kwargs"].get("column", "table")
    succes = "✅ OK" if r["success"] else "❌ ÉCHEC"
    detail = ""
    if "unexpected_percent" in r["result"]:
        detail = f"{r['result']['unexpected_percent']:.2f}% valeurs inattendues"
    print(f"{expectation_type} [{colonne}]  {succes}  {detail}")

In [0]:

# ============================================================
# Sauvegarde du rapport de validation dans le Volume UC
# Permet de conserver une trace horodatée de chaque validation
# ============================================================


rapport = {
    "date_validation": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "table": TABLES['silver'],
    "version_gx": gx.__version__,
    "echantillon": {
        "nb_lignes": 10000,
        "expectations_evaluees": stats['evaluated_expectations'],
        "expectations_reussies": stats['successful_expectations'],
        "taux_succes": stats['success_percent']
    },
    "echantillon_stratifie": {
        "nb_lignes": len(df_full_pandas),
        "expectations_evaluees": stats_full['evaluated_expectations'],
        "expectations_reussies": stats_full['successful_expectations'],
        "taux_succes": stats_full['success_percent']
    },
    "succes_global": bool(results_full.success)
}

chemin_rapport = "/Volumes/workspace/bronze/raw_data/rapport_validation_gx.json"

with open(chemin_rapport, "w", encoding="utf-8") as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

print(f"Rapport sauvegardé : {chemin_rapport}")
print(json.dumps(rapport, ensure_ascii=False, indent=2))
